# Data Understanding – Gutenberg Gait Database

Explorative Analyse für das Data-Understanding-Kapitel:
1. Altersverteilung
2. Geschlechterverteilung
3. Körpergewichtsverteilung (`BODY_MASS`)
4. Bodenreaktionskräfte (vertikal / anterior-posterior / medio-lateral) über den Gangzyklus – linker Fuß, rechter Fuß, und ein direkter Vergleich beider Seiten

In [13]:
#################
#Imports
#################

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#################
#Configuration
#################

# Dateipfade
DATA_PATH = 'GutenbergGaitDatabase/'
META_FILE = 'GRF_metadata.csv'
OUTPUT_DIR = 'figures'
DPI = 300

# Dateien der drei Kraftkomponenten
V_LEFT_FILE = 'GRF_F_V_PRO_left.csv'
AP_LEFT_FILE = 'GRF_F_AP_PRO_left.csv'
ML_LEFT_FILE = 'GRF_F_ML_PRO_left.csv'

# Altersgruppen
AGE_BINS = [0, 30, 45, 60, 100]
AGE_LABELS = ['18-30', '31-45', '46-60', '61+']

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
#################
# Load Data
#################

# Linke Kraftkurven laden
v_left = pd.read_csv(os.path.join(DATA_PATH, V_LEFT_FILE))
ap_left = pd.read_csv(os.path.join(DATA_PATH, AP_LEFT_FILE))
ml_left = pd.read_csv(os.path.join(DATA_PATH, ML_LEFT_FILE))

# Metadaten laden und an die V-Datei anhängen
metadata = pd.read_csv(os.path.join(DATA_PATH, META_FILE))

merge_cols = [
    c for c in ['DATASET_ID', 'SUBJECT_ID', 'SESSION_ID']
    if c in v_left.columns and c in metadata.columns
]

v_left = v_left.merge(metadata, on=merge_cols, how='left')

print(f'V:  {v_left.shape[0]} Trials')
print(f'AP: {ap_left.shape[0]} Trials')
print(f'ML: {ml_left.shape[0]} Trials')

# Eine Zeile pro Person für die demografische Statistik
subj_df = (
    v_left[['SUBJECT_ID', 'AGE', 'SEX', 'HEIGHT', 'BODY_MASS']]
    .drop_duplicates('SUBJECT_ID')
    .reset_index(drop=True)
)

print(f'Anzahl eindeutiger Subjekte: {len(subj_df)}')

v_left.head()


In [ ]:
#################
#Zusammenfassung der demografischen Daten
#################

n = len(subj_df)
ages = subj_df['AGE'].dropna()
gender = subj_df['SEX'].dropna()
weight = subj_df['BODY_MASS'].dropna()

print(f'Anzahl Subjekte: {n}')
print(f'Alter: M={ages.mean():.1f}, SD={ages.std():.1f}, '
      f'Min={ages.min():.0f}, Max={ages.max():.0f}')

print('\nAltersgruppen:')
print(
    pd.cut(ages, bins=AGE_BINS, labels=AGE_LABELS, include_lowest=True)
    .value_counts()
    .sort_index()
)

print('\nGeschlecht (n / %):')
counts = gender.value_counts()
for sex, count in counts.items():
    print(f'  {sex}: {count} ({count / len(gender) * 100:.1f}%)')

print(f'\nKörpergewicht: M={weight.mean():.1f} kg, SD={weight.std():.1f} kg, '
      f'Min={weight.min():.1f} kg, Max={weight.max():.1f} kg')


In [ ]:
#################
# Age distrobution
#################

fig, ax = plt.subplots(figsize=(6, 4.5))

ax.hist(ages, bins=20, edgecolor='white')
ax.axvline(
    ages.mean(),
    linestyle='--',
    linewidth=1,
    label=f'Mean = {ages.mean():.1f} years'
)

ax.set_title('Age Distribution')
ax.set_xlabel('Age (years)')
ax.set_ylabel('Number of Subjects')
ax.legend(frameon=False)

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'age_distribution.png'),
            dpi=DPI, bbox_inches='tight')
plt.show()


In [ ]:
#################
# SEX distrobution
#################

gender_counts = gender.value_counts()
gender_pct = gender_counts / gender_counts.sum() * 100

fig, ax = plt.subplots(figsize=(6, 4.5))
bars = ax.bar(gender_counts.index.astype(str), gender_counts.values)

for bar, sex in zip(bars, gender_counts.index):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f'{gender_counts[sex]}\n({gender_pct[sex]:.1f}%)',
        ha='center',
        va='bottom'
    )

ax.set_title('Sex Distribution')
ax.set_xlabel('Sex')
ax.set_ylabel('Number of Subjects')
ax.margins(y=0.15)

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'sex_distribution.png'),
            dpi=DPI, bbox_inches='tight')
plt.show()


In [ ]:
#################
# GRF (all directions, left foot)
#################



def get_curve(df, prefix):
    cols = [c for c in df.columns if c.startswith(prefix)]
    cols.sort(key=lambda c: int(c.replace(prefix, "")))
    
    arr = df[cols].to_numpy(dtype=float)
    mean = arr.mean(axis=0)
    std = arr.std(axis=0)
    x = np.arange(len(cols)) / (len(cols) - 1) * 100
    
    return x, mean, std

fig, ax = plt.subplots(figsize=(7, 5))

for df, prefix, label in [
    (v_left, 'F_V_PRO_', 'Vertical (V)'),
    (ap_left, 'F_AP_PRO_', 'Anterior-Posterior (AP)'),
    (ml_left, 'F_ML_PRO_', 'Medio-Lateral (ML)')
]:
    x, mean, std = get_curve(df, prefix)
    ax.plot(x, mean, label=label, linewidth=2)
    ax.fill_between(x, mean - std, mean + std, alpha=0.15)

ax.axhline(0, linewidth=0.8, linestyle=':')
ax.set_title('Ground Reaction Forces over Gait Cycle (Left Foot)')
ax.set_xlabel('Gait Cycle (%)')
ax.set_ylabel('Force (normalized to body weight)')
ax.legend(frameon=False)

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'grf_left_v_ap_ml.png'),
            dpi=DPI, bbox_inches='tight')
plt.show()


In [ ]:
#################
# GRF (all directions, right foot)
#################

v_right = pd.read_csv(os.path.join(DATA_PATH, 'GRF_F_V_PRO_right.csv'))
ap_right = pd.read_csv(os.path.join(DATA_PATH, 'GRF_F_AP_PRO_right.csv'))
ml_right = pd.read_csv(os.path.join(DATA_PATH, 'GRF_F_ML_PRO_right.csv'))

fig, ax = plt.subplots(figsize=(7, 5))

for df, prefix, label in [
    (v_right, 'F_V_PRO_', 'Vertical (V)'),
    (ap_right, 'F_AP_PRO_', 'Anterior-Posterior (AP)'),
    (ml_right, 'F_ML_PRO_', 'Medio-Lateral (ML)')
]:
    x, mean, std = get_curve(df, prefix)
    ax.plot(x, mean, label=label, linewidth=2)
    ax.fill_between(x, mean - std, mean + std, alpha=0.15)

ax.axhline(0, linewidth=0.8, linestyle=':')
ax.set_title('Ground Reaction Forces over Gait Cycle (Right Foot)')
ax.set_xlabel('Gait Cycle (%)')
ax.set_ylabel('Force (normalized to body weight)')
ax.legend(frameon=False)

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'grf_right_v_ap_ml.png'),
            dpi=DPI, bbox_inches='tight')
plt.show()
